In [ ]:
# yt-dlp: URLから形式を選んでダウンロード（このノートブックは1セルだけで実行できます）
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

# 未インストールの場合だけ yt-dlp をインストールします。
if importlib.util.find_spec("yt_dlp") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "yt-dlp"])

import yt_dlp

url = input("動画またはプレイリストのURL: ").strip()
if not url:
    raise ValueError("URLを入力してください。")

choices = {
    "mp4": {"format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best", "postprocessors": []},
    "webm": {"format": "bestvideo[ext=webm]+bestaudio[ext=webm]/best[ext=webm]/best", "postprocessors": []},
    "mp3": {"format": "bestaudio/best", "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}]},
    "m4a": {"format": "bestaudio[ext=m4a]/bestaudio", "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "m4a"}]},
}
print("選択できる拡張子: " + ", ".join(choices))
extension = input("拡張子 [mp4]: ").strip().lower() or "mp4"
if extension not in choices:
    raise ValueError(f"未対応の拡張子です: {extension}（mp4 / webm / mp3 / m4a から選択してください）")

def safe_name(name):
    name = re.sub(r'[\\/:*?"<>|\x00-\x1f]', "_", str(name)).strip().rstrip(".")
    return name[:180] or "playlist"

# 先に情報だけ取得して、プレイリストかどうかと名前を判定します。
with yt_dlp.YoutubeDL({"quiet": True, "no_warnings": True, "skip_download": True}) as probe:
    info = probe.extract_info(url, download=False)

is_playlist = info.get("_type") == "playlist" or (info.get("entries") is not None and info.get("webpage_url") is None)
settings = choices[extension]
common = {
    "format": settings["format"],
    "merge_output_format": extension if extension in {"mp4", "webm"} else None,
    "postprocessors": settings["postprocessors"],
    "noplaylist": not is_playlist,
    "windowsfilenames": True,
    "quiet": False,
}
common = {k: v for k, v in common.items() if v is not None}

if is_playlist:
    playlist_name = safe_name(info.get("title", "playlist"))
    with tempfile.TemporaryDirectory(prefix="yt_dlp_") as temp_dir:
        common["outtmpl"] = str(Path(temp_dir) / "%(playlist_index)03d - %(title)s.%(ext)s")
        with yt_dlp.YoutubeDL(common) as ydl:
            ydl.download([url])
        zip_path = Path.cwd() / f"{playlist_name}.zip"
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for file_path in sorted(Path(temp_dir).rglob("*")):
                if file_path.is_file() and not file_path.name.endswith(".part"):
                    archive.write(file_path, arcname=file_path.name)
    print(f"プレイリストを保存しました: {zip_path}")
else:
    common["outtmpl"] = str(Path.cwd() / "%(title)s.%(ext)s")
    with yt_dlp.YoutubeDL(common) as ydl:
        ydl.download([url])
    print("動画を保存しました。")

# mp4/webm の結合や mp3/m4a 変換には ffmpeg が必要です。
